# Raw gauge converter

`C:/Users/_s2220186/Desktop/Julia/gauge` 안의 모든 `.jld2` gauge 파일을 학습용 raw binary `.gbin` 파일로 변환합니다.

In [ ]:
using Pkg
Pkg.activate("C:/Users/_s2220186/Desktop/Julia/LMP")
using Gaugefields
using LMP
using JLD2
using CUDA

  Activating project at `C:\Users\_s2220186\Desktop\Julia\LMP`
[ Info: Precompiling LMP [a75e356b-bbea-40ef-9c9d-31e1efba0018] 
┌ Info: Skipping precompilation due to precompilable error. Importing LMP [a75e356b-bbea-40ef-9c9d-31e1efba0018].
└   exception = Error when precompiling module, potentially caused by a __precompile__(false) declaration in the module.
┌ Error: You are using CUDA 13.1.0, but CUDA.jl was precompiled for CUDA 13.0.0.
│ This is unexpected; please file an issue.
└ @ CUDA C:\Users\_s2220186\.julia\packages\CUDA\724Sm\src\initialization.jl:148
[ Info: Threads backend loaded with 1 threads


`U_buf`는 기존 학습 노트북에서 `train`/`train_timed`에 넘기던 GPU gauge buffer입니다. 변환할 때도 같은 shape/blockinfo가 필요합니다.

In [2]:
gauge_dir = raw"C:\Users\_s2220186\Desktop\Julia\gauge"
raw_dir = joinpath(gauge_dir, "raw_gauge")

gauge_paths = sort(filter(p -> endswith(lowercase(p), ".jld2"),
                          joinpath.(gauge_dir, readdir(gauge_dir))))
U_buf = Initialize_Gaugefields(NC, Nwing, NX, NY, NZ, NT, condition="hot")
@assert !isempty(gauge_paths) "gauge_dir 안에 .jld2 파일이 없습니다: $gauge_dir"
@assert isdefined(Main, :U_buf) "U_buf가 필요합니다. 기존 학습 노트북의 U_buf 생성 셀을 먼저 실행하세요."

println("gauge_dir = ", gauge_dir)
println("raw_dir   = ", raw_dir)
println("num gauges = ", length(gauge_paths))
first(gauge_paths, min(5, length(gauge_paths)))

LoadError: UndefVarError: `Initialize_Gaugefields` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
Hint: a global variable of this name may be made accessible by importing Gaugefields in the current active module Main

In [ ]:
raw_paths = LMP.Layer_sequence.convert_gauge_paths_to_raw(
    gauge_paths,
    U_buf;
    raw_dir=raw_dir,
    force=false,
)

println("converted gauges: ", length(raw_paths))
first(raw_paths, min(5, length(raw_paths)))

In [ ]:
JLD2.jldsave(joinpath(gauge_dir, "raw_gauge_paths.jld2"); raw_paths)
println("saved: ", joinpath(gauge_dir, "raw_gauge_paths.jld2"))

변환 후 학습에서는 `raw_paths`와 `gauge_format=:raw`를 사용합니다.

In [ ]:
# 예시
# to = train_timed(opt, layers, raw_paths, U_buf, params; epochs=10, batch=1, gauge_format=:raw)
# train(opt, layers, raw_paths, U_buf, params; epochs=10, batch=1, gauge_format=:raw)